In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from src.entropy_metrics import compute_all_metrics


WINDOW_SIZE = 500
STEP = 100

os.makedirs('results/figures', exist_ok=True)
os.makedirs('results/logs', exist_ok=True)

In [ ]:
import os

print(f"Directory di lavoro attuale: {os.getcwd()}")

df = pd.read_csv('data/baseline_dataset.csv')
signal = df['sensor_value'].values

# Calcolo entropia su finestre mobili
entropy_history = []
for i in range(0, len(signal) - WINDOW_SIZE, STEP):
    window = signal[i : i + WINDOW_SIZE]
    metrics = compute_all_metrics(window)
    entropy_history.append(metrics)

df_h = pd.DataFrame(entropy_history)
print("Calcolo completato per", len(df_h), "finestre temporali.")

In [ ]:
baseline_stats = {
    'shannon_mean': df_h['shannon'].mean(),
    'shannon_std': df_h['shannon'].std(),
    'sample_en_mean': df_h['sample_en'].mean(),
    'sample_en_std': df_h['sample_en'].std()
}

with open('results/logs/baseline_metrics.json', 'w') as f:
    json.dump(baseline_stats, f)

print(f"Baseline stabilita: Shannon Mean = {baseline_stats['shannon_mean']:.4f}")

In [ ]:
plt.figure(figsize=(12, 5))

# Plot 1: Serie temporale dell'entropia
plt.subplot(1, 2, 1)
plt.plot(df_h['shannon'], color='blue', label='Shannon H')
plt.axhline(baseline_stats['shannon_mean'], color='red', linestyle='--', label='Mean')
plt.title("H(k) Stability - Baseline")
plt.xlabel("Window Index")
plt.ylabel("Entropy")
plt.legend()

# Plot 2: Distribuzione (Istogramma)
plt.subplot(1, 2, 2)
sns.histplot(df_h['shannon'], kde=True, color='green')
plt.title("Entropy Distribution")

plt.tight_layout()
plt.savefig('results/figures/baseline_stability.png')
plt.show()